In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

# searchselect

A searchable multi-select picker for any list of strings. It knows nothing but strings — column names, category values, file paths, whatever you have a list of.

In [ ]:
import random
import string

from searchselect import SearchSelect


def randomword(length: int = 10) -> str:
    letters = string.ascii_lowercase + string.ascii_uppercase
    return "".join(random.choice(letters) for _ in range(length))


test_items = [f"Item{i}" for i in range(1, 100 + 1)] + [randomword() for _ in range(100)]
random.shuffle(test_items)

picker = SearchSelect(items=test_items, theme="light")
picker

## Reading the result

`selected` is an **explicit choice** — the items you ticked. Tick a few above, then re-run this cell.

Filtering never changes it: tick something, then type a search that excludes it, and it stays selected. The search box changes what you can *see*, not what you have *chosen*.

In [ ]:
picker.selected

`filtered` is a **query result** — whatever the search box currently matches. With an empty search box it is every item, never an empty list.

That makes it a bulk-select: type a pattern, take all the matches, no clicking.

In [ ]:
len(picker.filtered), picker.filtered[:5]

## Searching

`query` is writable, so you can drive the search box from Python. Matching runs in the kernel, so this applies immediately — `filtered` is correct in the same cell.

The header checkbox selects **everything matching the current query**, not just what's on screen, so query-plus-select-all is the click-free path to a large selection.

In [ ]:
picker.query = "Item1"
len(picker.filtered), picker.filtered[:5]

Because matching is Python's `re`, the pattern you type is a **Python** regex — including syntax a JavaScript engine would reject, like named groups. Whatever you craft in the box works verbatim in your own code.

In [ ]:
import re

picker.regex = True
picker.query = r"^Item(?P<n>\d{2})$"

# the widget's matches, and the same pattern applied by hand, agree exactly
by_hand = [i for i in picker.items if re.search(picker.query, i)]
picker.filtered == by_hand, len(by_hand)

An unparseable pattern matches nothing and says why, in Python's own words — which is a good deal more useful than a bare "invalid".

In [ ]:
picker.query = "Item("
picker.query_error, picker.filtered

In [ ]:
picker.query = ""
picker.regex = False
len(picker.filtered)  # empty query means every item

## Writing back

`selected` is writable, so you can preselect or clear it from Python. The checkboxes follow. Values that aren't in `items` are ignored.

In [ ]:
picker.selected = ["Item1", "Item2", "not-a-real-item"]
picker.selected

In [ ]:
picker.selected = []
picker.selected

Selection is keyed on the item string, not on its row position. So replacing `items` keeps ticks on items that still exist and drops the rest — it never silently transfers a tick to whatever moved into that row.

In [ ]:
picker.selected = ["Item1", "Item2"]
picker.items = ["Item2", "Item3"]

picker.selected  # Item1 is gone, Item2 survived

## Using it

The widget has no dataframe dependency — it hands back a plain `list[str]`, so it composes with whatever you use. Picking columns out of a wide frame is the obvious case.

In [ ]:
import polars as pl

df = pl.DataFrame({name: [1, 2, 3] for name in test_items[:20]})

columns = SearchSelect(items=df.columns)
columns

In [ ]:
df.select(columns.selected) if columns.selected else df.head()